# Step 1: Import Libraries

In [70]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

# Step 2: Load Dataset

In [71]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

# Step 3: Merge Datasets

In [72]:
movies = movies.merge(credits,on='title')

In [73]:
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


# 
Step 4: Select Important Features

In [74]:
movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew']]

# Step 5: Handle Missing Values

In [75]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [76]:
movies.dropna(inplace=True)

# Step 6: Convert JSON-like Strings

### Function

In [77]:
import ast

def convert(text):
    L=[]
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L
    

### Apply:

In [78]:
movies['genres']=movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)

# Step 7: Extract Top 3 Cast Members

In [79]:
def convert_cast(text):
    L=[]
    counter=0

    for i in ast.literal_eval(text):
        if counter !=3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [80]:
movies['cast'] = movies['cast'].apply(convert_cast)

# Step 8: Extract Director

In [81]:
def fetch_director(text):
    L = []

    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L
    

In [82]:
movies['crew'] = movies['crew'].apply(fetch_director)

# Step 9: Convert Overview into List

In [83]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

# Step 10: Remove Spaces

### Function:

In [84]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

### Apply:

In [85]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

# Step 11: Create Tags

### Combine all features

In [86]:
movies['tags'] = movies['overview'] + \
                movies['genres'] + \
                movies['keywords'] + \
                movies['cast'] + \
                movies['crew']

# Step 12: Create New DataFrame

In [87]:
new_df = movies[['movie_id','title','tags']]

In [88]:
new_df

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...
4804,9367,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui..."
4805,72766,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended..."
4806,231617,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,..."
4807,126186,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is..."


# Step 13: Convert List to String

In [89]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [90]:
new_df['tags']

0       In the 22nd century, a paraplegic Marine is di...
1       Captain Barbossa, long believed to be dead, ha...
2       A cryptic message from Bond’s past sends him o...
3       Following the death of District Attorney Harve...
4       John Carter is a war-weary, former military ca...
                              ...                        
4804    El Mariachi just wants to play his guitar and ...
4805    A newlywed couple's honeymoon is upended by th...
4806    "Signed, Sealed, Delivered" introduces a dedic...
4807    When ambitious New York attorney Sam is sent t...
4808    Ever since the second grade when he first saw ...
Name: tags, Length: 4806, dtype: object

# Step 14: Lowercase

In [91]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [92]:
new_df['tags']

0       in the 22nd century, a paraplegic marine is di...
1       captain barbossa, long believed to be dead, ha...
2       a cryptic message from bond’s past sends him o...
3       following the death of district attorney harve...
4       john carter is a war-weary, former military ca...
                              ...                        
4804    el mariachi just wants to play his guitar and ...
4805    a newlywed couple's honeymoon is upended by th...
4806    "signed, sealed, delivered" introduces a dedic...
4807    when ambitious new york attorney sam is sent t...
4808    ever since the second grade when he first saw ...
Name: tags, Length: 4806, dtype: object

# Step 15: Stemming

In [93]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    y=[]

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [94]:
new_df['tags'] = new_df['tags'].apply(stem)

In [95]:
new_df['tags']

0       in the 22nd century, a parapleg marin is dispa...
1       captain barbossa, long believ to be dead, ha c...
2       a cryptic messag from bond’ past send him on a...
3       follow the death of district attorney harvey d...
4       john carter is a war-weary, former militari ca...
                              ...                        
4804    el mariachi just want to play hi guitar and ca...
4805    a newlyw couple' honeymoon is upend by the arr...
4806    "signed, sealed, delivered" introduc a dedic q...
4807    when ambiti new york attorney sam is sent to s...
4808    ever sinc the second grade when he first saw h...
Name: tags, Length: 4806, dtype: object

# Step 16: Vectorization

### Using CountVectorizer

In [96]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = tfidf.fit_transform(new_df['tags']).toarray()

# Step 17: Cosine Similarity

In [97]:
from sklearn.metrics.pairwise import cosine_similarity

In [98]:
similarity = cosine_similarity(vectors)

In [99]:
similarity.shape

(4806, 4806)

# Step 18: Recommendation Function

In [100]:
def recommend(movie):

    matches = new_df[new_df['title'] == movie]

    if matches.empty:
        print(f"Movie '{movie}' not found!")
        return

    movie_index = matches.index[0]

    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

### Add a search helper

- To find movies even when you don't know the exact title:

In [101]:
def search_movie(keyword):
    results = new_df[
        new_df['title'].str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    return results['title'].tolist()

# Step 19: Test

In [102]:
recommend('Avatar')

Aliens
Falcon Rising
Battle: Los Angeles
Aliens vs Predator: Requiem
Apollo 18


In [103]:
movie_index = new_df[new_df['title'] == 'Avatar'].index[0]

distances = similarity[movie_index]

movies_list = sorted(
    list(enumerate(distances)),
    key=lambda x: x[1],
    reverse=True
)[1:11]

for i in movies_list:
    print(new_df.iloc[i[0]].title, "->", round(i[1],3))

Aliens -> 0.252
Falcon Rising -> 0.226
Battle: Los Angeles -> 0.194
Aliens vs Predator: Requiem -> 0.188
Apollo 18 -> 0.177
Star Trek Into Darkness -> 0.173
Meet Dave -> 0.166
Predators -> 0.163
Titan A.E. -> 0.156
The Book of Life -> 0.155


In [104]:
recommend("The Dark Knight")

The Dark Knight Rises
Batman Returns
Batman Begins
Batman Forever
Batman: The Dark Knight Returns, Part 2


In [105]:
recommend("Titanic")

Ghost Ship
Poseidon
Pirates of the Caribbean: On Stranger Tides
The Rose
Dear Frankie


In [106]:
recommend("Toy Story")

Toy Story 3
Toy Story 2
Small Soldiers
For Your Consideration
The 41–Year–Old Virgin Who Knocked Up Sarah Marshall and Felt Superbad About It


In [107]:
recommend("Iron Man")

Iron Man 2
Iron Man 3
Avengers: Age of Ultron
Ant-Man
Captain America: Civil War


In [108]:
recommend("Harry Potter and the Sorcerer's Stone")

Movie 'Harry Potter and the Sorcerer's Stone' not found!


In [109]:

search_movie("Harry")

['Harry Potter and the Half-Blood Prince',
 'Harry Potter and the Order of the Phoenix',
 'Harry Potter and the Goblet of Fire',
 'Harry Potter and the Prisoner of Azkaban',
 "Harry Potter and the Philosopher's Stone",
 'Harry Potter and the Chamber of Secrets',
 'Dumb and Dumberer: When Harry Met Lloyd',
 'Deconstructing Harry',
 'When Harry Met Sally...',
 'Harry Brown',
 'The Trouble with Harry']

In [110]:
search_movie("spider")

['Spider-Man 3',
 'The Amazing Spider-Man',
 'Spider-Man 2',
 'The Amazing Spider-Man 2',
 'Spider-Man',
 'The Spiderwick Chronicles',
 'Along Came a Spider',
 'Spider',
 'Kingdom of the Spiders']

# Step 20: Save Model

In [111]:
import pickle

### Save movie dataframe

In [112]:
pickle.dump(
    new_df,
    open('movies.pkl','wb')
)

### Save similarity matrix

In [113]:
pickle.dump(
    similarity,
    open('similarity.pkl','wb')
)